## Setup & Dataset

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt


from dotenv import load_dotenv
from datasets import load_dataset
from openai import OpenAI

load_dotenv(override=True)

dataset = load_dataset(
    "Carson-Shively/used-car-price",
    split="train"
)

print(f"Number of cars: {len(dataset)}")

## Data Cleaning

In [ ]:
def clean_price(example):
    example["price"] = int(
        str(example["price"])
        .replace(",", "")
        .replace("$", "")
    )
    return example


dataset = dataset.map(clean_price)

In [ ]:
def clean_category(example):

    if example["fuel_type"] is None or example["fuel_type"] in ["–", "not supported"]:
        example["fuel_type"] = "unknown"
    else:
        example["fuel_type"] = example["fuel_type"].lower()

    if example["accident"] is None:
        example["accident"] = "unknown"
    else:
        example["accident"] = example["accident"].lower()

    if example["clean_title"] is None:
        example["clean_title"] = "unknown"
    else:
        example["clean_title"] = example["clean_title"].lower()

    return example


dataset = dataset.map(clean_category)

In [ ]:
def clean_milage(example):
    if example["milage"] is not None:
        example["milage"] = int(
            str(example["milage"])
            .replace(",", "")
            .replace(" mi.", "")
        )
    return example


dataset = dataset.map(clean_milage)

In [ ]:
print(f"Number of cars after cleaning: {dataset.num_rows}")
print(f"Missing model years: {sum(x is None for x in dataset['model_year'])}")

## Evaluation

In [ ]:
samples = dataset.shuffle(seed=42).select(range(200))

print(f"Evaluation samples: {len(samples)}")

## GPT Prediction

In [ ]:
client = OpenAI()

In [ ]:
def format_message(item):
    return {
        "system": (
            "You are a car pricing expert. "
            "Estimate the price of the used car based on its specifications. "
            "Return only one estimated price in USD."
        ),
        "user": (
            f"Brand: {item['brand']}\n"
            f"Model: {item['model']}\n"
            f"Year: {item['model_year']}\n"
            f"Mileage: {item['milage']}\n"
            f"Fuel: {item['fuel_type']}\n"
            f"Engine: {item['engine']}\n"
            f"Transmission: {item['transmission']}\n"
            f"Exterior: {item['ext_col']}\n"
            f"Interior: {item['int_col']}\n"
            f"Accident: {item['accident']}\n"
            f"Clean title: {item['clean_title']}"
        )
    }

In [ ]:
def predict_price(item):
    message = format_message(item)

    response = client.responses.create(
        model="gpt-5",
        instructions=message["system"],
        input=message["user"]
    )

    return response.output_text

## Evaluation

In [ ]:
results = []

for i, item in enumerate(samples):

    prediction = predict_price(item)

    prediction = (
        prediction
        .replace("$", "")
        .replace(",", "")
        .replace("USD", "")
        .strip()
    )

    gpt_price = float(prediction)
    real_price = item["price"]

    error = abs(gpt_price - real_price)
    error_percent = (error / real_price) * 100

    results.append({
        "real_price": real_price,
        "gpt_prediction": gpt_price,
        "error": error,
        "error_percent": error_percent
    })

    print(f"{i + 1}/200 done")

In [ ]:
results_df = pd.DataFrame(results)

mae = results_df["error"].mean()
mape = results_df["error_percent"].mean()

print(f"MAE: ${mae:,.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:


plt.figure(figsize=(8, 6))

plt.scatter(
    results_df["real_price"],
    results_df["gpt_prediction"]
)

plt.xlabel("Real Price")
plt.ylabel("GPT Prediction")
plt.title("Real Price vs GPT Prediction")

plt.show()